# Pupil analysis v2: session-z P5 minus P4

Run Blocks A1-A10 in order. Train groups use fixed original trial numbers. Test A and C are compared with strict immediately preceding B trials, followed by 500-repeat chronology-stratified B-selection sensitivity analysis. Fractional-response v1 is archived and is not used here.

In [ ]:
# Block A1 - imports, plotting defaults, and compact plotting helpers
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from attention_alignment import load_config
from attention_alignment.pupil_analysis import (
    add_original_trial_groups, extract_item_trials, extract_sequence_trials,
    infer_stimulus_window, pair_immediate_preceding_b, p5_minus_p4,
    plot_first_last_group_traces, plot_grouped_trial_traces, plot_metric_trend,
    plot_trial_heatmap, pupil_qc_table, resampled_trace_effects,
    save_figure_bundle, sequence_stimulus_intervals, stratified_b_resampling,
    trial_metrics,
)

plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': False, 'font.size': 10,
})
CONDITION_COLORS = {'AAAAB': '#D55E00', 'AAAAA': '#0072B2', 'AAAAC': '#009E73'}

def display_figure_once(figure):
    display(figure)
    plt.close(figure)

def trace_summary(table, value_column, error_band='sem'):
    summary = table.groupby('trial_time_s')[value_column].agg(['mean', 'std', 'count']).reset_index()
    width = summary['std'] / np.sqrt(summary['count'].clip(lower=1))
    if error_band == 'ci95':
        width = 1.96 * width
    elif error_band != 'sem':
        raise ValueError("ERROR_BAND must be 'sem' or 'ci95'")
    summary['lower'] = summary['mean'] - width.fillna(0.0)
    summary['upper'] = summary['mean'] + width.fillna(0.0)
    return summary

def plot_adjacent_pair(traces, pairs, catch_pattern, value_column, stimulus_s, error_band):
    if pairs.empty:
        raise ValueError(f'No strict adjacent-B pairs are available for {catch_pattern}')
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    definitions = [
        (pairs['reference_trial_id'], pairs['reference_label'].iloc[0], '#D55E00'),
        (pairs['catch_trial_id'], catch_pattern, CONDITION_COLORS[catch_pattern]),
    ]
    for trial_ids, label, color in definitions:
        subset = traces.loc[traces['trial_id'].isin(trial_ids)]
        summary = trace_summary(subset, value_column, error_band)
        axes[0].fill_between(summary['trial_time_s'], summary['lower'], summary['upper'], color=color, alpha=0.18, linewidth=0)
        axes[0].plot(summary['trial_time_s'], summary['mean'], color=color, linewidth=2, label=f'{label} (n={len(trial_ids)})')
    axes[0].axvspan(0, stimulus_s, color='#E5E5E5', alpha=0.5, zorder=0)
    axes[0].axvline(0, color='black', linewidth=0.8)
    axes[0].axhline(0, color='#777777', linewidth=0.6)
    axes[0].set(xlabel='Time from P5 onset (s)', ylabel='P5 - P4 pupil session z', title=f'{catch_pattern} vs strict adjacent B')
    axes[0].legend(frameon=False)
    for row in pairs.itertuples(index=False):
        axes[1].plot([0, 1], [row.reference_response, row.catch_response], color='#B8B8B8', linewidth=0.9, alpha=0.8)
    axes[1].scatter(np.zeros(len(pairs)), pairs['reference_response'], color='#D55E00', s=30, zorder=3)
    axes[1].scatter(np.ones(len(pairs)), pairs['catch_response'], color=CONDITION_COLORS[catch_pattern], s=30, zorder=3)
    axes[1].axhline(0, color='#777777', linewidth=0.6)
    axes[1].set(xticks=[0, 1], xticklabels=[pairs['reference_label'].iloc[0], catch_pattern], ylabel='Mean response during P5', title=f'Paired effect; mean difference={pairs["response_difference"].mean():.3f}')
    figure.tight_layout()
    return figure

def plot_resampling_sensitivity(effects, trace_effect, catch_pattern, stimulus_s):
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(effects['response_difference'], bins=25, color=CONDITION_COLORS[catch_pattern], alpha=0.75)
    axes[0].axvline(0, color='#777777', linewidth=0.8)
    axes[0].axvline(effects['response_difference'].mean(), color='black', linewidth=2)
    axes[0].set(xlabel=f'{catch_pattern} - sampled B mean response', ylabel='Repeats', title='500-repeat B-selection sensitivity')
    axes[1].fill_between(trace_effect['trial_time_s'], trace_effect['b_selection_interval_low'], trace_effect['b_selection_interval_high'], color=CONDITION_COLORS[catch_pattern], alpha=0.2, linewidth=0)
    axes[1].plot(trace_effect['trial_time_s'], trace_effect['mean'], color=CONDITION_COLORS[catch_pattern], linewidth=2)
    axes[1].axvspan(0, stimulus_s, color='#E5E5E5', alpha=0.5, zorder=0)
    axes[1].axvline(0, color='black', linewidth=0.8)
    axes[1].axhline(0, color='#777777', linewidth=0.6)
    axes[1].set(xlabel='Time from P5 onset (s)', ylabel=f'{catch_pattern} - sampled B', title='B-selection interval (2.5-97.5%)')
    figure.tight_layout()
    return figure

CONFIG_PATH = Path('configs/sessions.local.yaml')
CONFIG = load_config(CONFIG_PATH)

In [ ]:
# Block A2 - parameters; edit only this block when switching sessions
ANALYSIS_VERSION = 'pupil03-session-z-v2.0.0'
SESSION_ID = 'replace_with_session_id'
CAMERA = '01'
TRAIN_GROUP_SIZE = 10
TEST_BLOCK_SIZE = 10
RESAMPLE_REPETITIONS = 500
RESAMPLE_SEED = 20260817
ERROR_BAND = 'sem'
SAMPLE_RATE_HZ = 30.0
MIN_TRIAL_VALID_FRACTION = 0.80
MIN_PHASE_VALID_FRACTION = 0.70
MAX_INTERPOLATION_GAP_S = 0.50
SAVE_RESULTS = True

SESSION_DIR = CONFIG.output_root / SESSION_ID
ANALYSIS_DIR = SESSION_DIR / 'pupil_analysis_v2_session_z'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
print({
    'version': ANALYSIS_VERSION, 'session': SESSION_ID, 'camera': CAMERA,
    'train_fixed_group_size': TRAIN_GROUP_SIZE,
    'test_fixed_block_size': TEST_BLOCK_SIZE,
    'resampling': (RESAMPLE_REPETITIONS, RESAMPLE_SEED),
    'analysis_dir': ANALYSIS_DIR,
})

In [ ]:
# Block A3 - load aligned inputs and enforce phase-level pupil QC
events = pd.read_parquet(SESSION_DIR / 'stimulus_events.parquet')
behavior = pd.read_parquet(SESSION_DIR / f'behavior_{CAMERA}.parquet')
SPEC = infer_stimulus_window(events)
qc = pupil_qc_table(behavior, events)
display(qc)
phase_qc = qc.set_index('phase')['pupil_valid_fraction']
READY = bool(
    phase_qc.get('train', 0) >= MIN_PHASE_VALID_FRACTION
    and phase_qc.get('test', 0) >= MIN_PHASE_VALID_FRACTION
)
print({'ready_for_train_and_test_analysis': READY, 'window_spec': SPEC})

def require_ready():
    if not READY:
        raise RuntimeError('Pupil QC failed. Re-run Notebook 02 and regenerate behavior Parquet.')

In [ ]:
# Block A4 - extract train sequence and primary P5-minus-P4 session-z response
require_ready()
train_sequence = extract_sequence_trials(
    events, behavior, phase='train', sequence_patterns=['AAAAB'],
    sample_rate_hz=SAMPLE_RATE_HZ, max_interpolation_gap_s=MAX_INTERPOLATION_GAP_S,
    min_valid_fraction=MIN_TRIAL_VALID_FRACTION,
)
train_sequence_metrics = trial_metrics(
    train_sequence, value_column='pupil_session_z', response_window_s=(0.0, SPEC.sequence_s)
)
train_p4_p5 = extract_item_trials(
    events, behavior, phase='train', item_positions=[4, 5], sequence_patterns=['AAAAB'],
    sample_rate_hz=SAMPLE_RATE_HZ, max_interpolation_gap_s=MAX_INTERPOLATION_GAP_S,
    min_valid_fraction=MIN_TRIAL_VALID_FRACTION,
)
train_p5_p4 = p5_minus_p4(train_p4_p5)
train_p5_p4_metrics = trial_metrics(
    train_p5_p4, value_column='p5_minus_p4_pupil_z',
    movement_column='p5_minus_p4_movement_z', response_window_s=(0.0, SPEC.stimulus_s),
)
train_p5_p4_metrics = add_original_trial_groups(train_p5_p4_metrics, TRAIN_GROUP_SIZE)
display(train_p5_p4_metrics[['sequence_index', 'trial_group', 'trial_valid', 'response_mean']].head(12))

In [ ]:
# Block A5 - train fixed-number groups and original trials 1-10 versus 91-100
sequence_intervals = sequence_stimulus_intervals('AAAAB', SPEC)
b_interval = [(0.0, SPEC.stimulus_s, 'B')]
fig_train_sequence_first_last = plot_first_last_group_traces(
    train_sequence, group_size=TRAIN_GROUP_SIZE, value_column='pupil_session_z',
    error_band=ERROR_BAND, stimulus_intervals=sequence_intervals, fixed_original_ranges=True,
)
fig_train_p5_p4_groups = plot_grouped_trial_traces(
    train_p5_p4, group_size=TRAIN_GROUP_SIZE, value_column='p5_minus_p4_pupil_z',
    stimulus_intervals=b_interval, columns=5, fixed_original_groups=True,
)
fig_train_p5_p4_heatmap = plot_trial_heatmap(
    train_p5_p4, value_column='p5_minus_p4_pupil_z'
)
fig_train_p5_p4_trend = plot_metric_trend(train_p5_p4_metrics, metric='response_mean')
fig_train_p5_p4_first_last = plot_first_last_group_traces(
    train_p5_p4, group_size=TRAIN_GROUP_SIZE, value_column='p5_minus_p4_pupil_z',
    error_band=ERROR_BAND, stimulus_intervals=b_interval, fixed_original_ranges=True,
)
for figure in (fig_train_sequence_first_last, fig_train_p5_p4_groups, fig_train_p5_p4_heatmap, fig_train_p5_p4_trend, fig_train_p5_p4_first_last):
    display_figure_once(figure)
last_train_index = int(train_p5_p4_metrics['sequence_index'].max())
train_endpoints = train_p5_p4_metrics.loc[
    train_p5_p4_metrics['sequence_index'].le(TRAIN_GROUP_SIZE)
    | train_p5_p4_metrics['sequence_index'].gt(last_train_index - TRAIN_GROUP_SIZE)
].copy()
train_endpoints['adaptation_stage'] = np.where(
    train_endpoints['sequence_index'].le(TRAIN_GROUP_SIZE), 'first_10', 'last_10'
)
train_endpoint_summary = train_endpoints.loc[train_endpoints['trial_valid']].groupby('adaptation_stage').agg(
    n=('trial_id', 'size'), response_mean=('response_mean', 'mean'), response_sem=('response_mean', 'sem'),
    movement_mean_z=('movement_mean_z', 'mean'),
).reset_index()
display(train_endpoint_summary)

In [ ]:
# Block A6 - extract test P4/P5 and calculate the primary response
test_patterns = ['AAAAB', 'AAAAA', 'AAAAC']
test_p4_p5 = extract_item_trials(
    events, behavior, phase='test', item_positions=[4, 5], sequence_patterns=test_patterns,
    sample_rate_hz=SAMPLE_RATE_HZ, max_interpolation_gap_s=MAX_INTERPOLATION_GAP_S,
    min_valid_fraction=MIN_TRIAL_VALID_FRACTION,
)
test_p5_p4 = p5_minus_p4(test_p4_p5)
test_p5_p4_metrics = trial_metrics(
    test_p5_p4, value_column='p5_minus_p4_pupil_z',
    movement_column='p5_minus_p4_movement_z', response_window_s=(0.0, SPEC.stimulus_s),
)
test_p5_p4_metrics = add_original_trial_groups(test_p5_p4_metrics, TEST_BLOCK_SIZE)
test_counts = test_p5_p4_metrics.groupby('sequence_pattern').agg(
    total_trials=('trial_id', 'size'), valid_trials=('trial_valid', 'sum')
).reset_index()
display(test_counts)

In [ ]:
# Block A7 - strict adjacent B_A and B_C paired comparisons
pair_tables, unmatched_tables, pair_figures = {}, {}, {}
for catch_pattern in ('AAAAA', 'AAAAC'):
    pairs, unmatched = pair_immediate_preceding_b(
        test_p5_p4_metrics, catch_pattern=catch_pattern
    )
    pair_tables[catch_pattern] = pairs
    unmatched_tables[catch_pattern] = unmatched
    pair_figures[catch_pattern] = plot_adjacent_pair(
        test_p5_p4, pairs, catch_pattern, 'p5_minus_p4_pupil_z', SPEC.stimulus_s, ERROR_BAND
    )
    display(pairs[['catch_sequence_index', 'reference_sequence_index', 'response_difference']])
    if not unmatched.empty:
        display(unmatched)
    display_figure_once(pair_figures[catch_pattern])
neighbor_pairs = pd.concat(pair_tables.values(), ignore_index=True)
unmatched_catches = pd.concat(unmatched_tables.values(), ignore_index=True)
neighbor_pair_summary = neighbor_pairs.groupby('comparison').agg(
    pairs=('pair_id', 'size'), mean_difference=('response_difference', 'mean'),
    sem_difference=('response_difference', 'sem'),
).reset_index()
display(neighbor_pair_summary)

In [ ]:
# Block A8 - 500-repeat B downsampling in fixed original 10-trial blocks
selection_tables, effect_tables, sensitivity_summaries = {}, {}, {}
repeat_trace_tables, trace_summaries, sensitivity_figures = {}, {}, {}
for seed_offset, catch_pattern in enumerate(('AAAAA', 'AAAAC')):
    selections, effects, summary = stratified_b_resampling(
        test_p5_p4_metrics, catch_pattern=catch_pattern, block_size=TEST_BLOCK_SIZE,
        repetitions=RESAMPLE_REPETITIONS, seed=RESAMPLE_SEED + seed_offset,
    )
    repeat_traces, trace_summary = resampled_trace_effects(
        test_p5_p4, selections, catch_pattern=catch_pattern
    )
    selection_tables[catch_pattern], effect_tables[catch_pattern] = selections, effects
    sensitivity_summaries[catch_pattern] = summary
    repeat_trace_tables[catch_pattern], trace_summaries[catch_pattern] = repeat_traces, trace_summary
    sensitivity_figures[catch_pattern] = plot_resampling_sensitivity(
        effects, trace_summary, catch_pattern, SPEC.stimulus_s
    )
    display(summary)
    display_figure_once(sensitivity_figures[catch_pattern])
resampling_selections = pd.concat(selection_tables.values(), ignore_index=True)
resampling_effects = pd.concat(effect_tables.values(), ignore_index=True)
resampling_summary = pd.concat(sensitivity_summaries.values(), ignore_index=True)
resampling_repeat_traces = pd.concat(repeat_trace_tables.values(), keys=repeat_trace_tables.keys(), names=['catch_pattern']).reset_index(level=0)
resampling_trace_summary = pd.concat(trace_summaries.values(), keys=trace_summaries.keys(), names=['catch_pattern']).reset_index(level=0)

In [ ]:
# Block A9 - compact baseline, movement, hull-correction, and outlier review
valid_test_metrics = test_p5_p4_metrics.loc[test_p5_p4_metrics['trial_valid']].copy()
qc_mean_columns = [
    'response_mean', 'movement_mean_z',
    'pupil_baseline_diameter_px_p4', 'pupil_baseline_diameter_px_p5',
    'pupil_hull_correction_fraction_p4', 'pupil_hull_correction_fraction_p5',
]
condition_qc = valid_test_metrics.groupby('sequence_pattern').agg(
    valid_trials=('trial_id', 'size'),
    **{column: (column, 'mean') for column in qc_mean_columns if column in valid_test_metrics},
).reset_index()
movement_response_correlation = valid_test_metrics.groupby('sequence_pattern').apply(
    lambda table: table['response_mean'].corr(table['movement_mean_z']) if len(table) >= 3 else np.nan,
    include_groups=False,
).rename('response_movement_correlation').reset_index()
top_absolute_response_trials = valid_test_metrics.assign(
    absolute_response=valid_test_metrics['response_mean'].abs()
).nlargest(5, 'absolute_response')
display(condition_qc)
display(movement_response_correlation)
display(top_absolute_response_trials[[
    'sequence_index', 'sequence_pattern', 'response_mean', 'movement_mean_z',
    *[column for column in qc_mean_columns if column.startswith('pupil_hull') and column in top_absolute_response_trials],
]])
print('Top-five trials are review targets only; this notebook does not exclude them automatically.')

In [ ]:
# Block A10 - save versioned tables, figures, and parameters
tables = {
    'session_qc': qc,
    'train_sequence_traces': train_sequence,
    'train_sequence_metrics': train_sequence_metrics,
    'train_p4_p5_traces': train_p4_p5,
    'train_p5_minus_p4_traces': train_p5_p4,
    'train_p5_minus_p4_metrics': train_p5_p4_metrics,
    'train_endpoints': train_endpoints,
    'train_endpoint_summary': train_endpoint_summary,
    'test_p4_p5_traces': test_p4_p5,
    'test_p5_minus_p4_traces': test_p5_p4,
    'test_p5_minus_p4_metrics': test_p5_p4_metrics,
    'test_counts': test_counts,
    'neighbor_pairs': neighbor_pairs,
    'unmatched_catches': unmatched_catches,
    'neighbor_pair_summary': neighbor_pair_summary,
    'resampling_selections': resampling_selections,
    'resampling_effects': resampling_effects,
    'resampling_summary': resampling_summary,
    'resampling_repeat_traces': resampling_repeat_traces,
    'resampling_trace_summary': resampling_trace_summary,
    'condition_qc': condition_qc,
    'movement_response_correlation': movement_response_correlation,
    'top_absolute_response_trials': top_absolute_response_trials,
}
figures = {
    'train_sequence_first_last_original_10': fig_train_sequence_first_last,
    'train_p5_minus_p4_fixed_groups': fig_train_p5_p4_groups,
    'train_p5_minus_p4_heatmap': fig_train_p5_p4_heatmap,
    'train_p5_minus_p4_response_trend': fig_train_p5_p4_trend,
    'train_p5_minus_p4_first_last_original_10': fig_train_p5_p4_first_last,
    **{f'test_{key}_strict_adjacent_pair': value for key, value in pair_figures.items()},
    **{f'test_{key}_resampling_sensitivity': value for key, value in sensitivity_figures.items()},
}
parameters = {
    'analysis_version': ANALYSIS_VERSION, 'session_id': SESSION_ID, 'camera': CAMERA,
    'primary_metric': 'mean(P5 pupil_session_z - P4 pupil_session_z)',
    'response_window_s': [0.0, SPEC.stimulus_s],
    'train_group_definition': 'fixed original sequence_index bins',
    'train_group_size': TRAIN_GROUP_SIZE,
    'test_reference_rule': 'only sequence_index - 1 when valid AAAAB',
    'test_resampling_block_size': TEST_BLOCK_SIZE,
    'resampling_repetitions': RESAMPLE_REPETITIONS, 'resampling_seed_A': RESAMPLE_SEED,
    'resampling_seed_C': RESAMPLE_SEED + 1, 'error_band': ERROR_BAND,
    'sample_rate_hz': SAMPLE_RATE_HZ, 'min_trial_valid_fraction': MIN_TRIAL_VALID_FRACTION,
    'min_phase_valid_fraction': MIN_PHASE_VALID_FRACTION,
    'max_interpolation_gap_s': MAX_INTERPOLATION_GAP_S,
    'condition_label_ms': SPEC.condition_label_ms, 'pre_stimulus_s': SPEC.pre_stimulus_s,
    'stimulus_s': SPEC.stimulus_s, 'sequence_s': SPEC.sequence_s,
}
if SAVE_RESULTS:
    ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        table.to_parquet(ANALYSIS_DIR / f'{name}.parquet', index=False)
        if len(table) <= 5000:
            table.to_csv(ANALYSIS_DIR / f'{name}.csv', index=False)
    for name, figure in figures.items():
        save_figure_bundle(figure, FIGURE_DIR / name)
    (ANALYSIS_DIR / 'analysis_parameters.json').write_text(
        json.dumps(parameters, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
    )
    print(f'Saved {len(tables)} tables and {len(figures)} figures to {ANALYSIS_DIR}')
else:
    print('SAVE_RESULTS=False: results remain only in notebook memory.')